In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')

# Parameters
BW = 125e3  # Bandwidth in Hz
fs = 2 * BW  # Sampling frequency
fixed_time = 0.035  # Fixed time window (35ms to show SF=12 fully)
SF_values = [7, 8, 9, 10, 11, 12]

def generate_upchirp(SF, BW, fs):
    """Generate single upchirp for given SF"""
    M = 2**SF
    Ts = M / BW
    N = int(fs * Ts)
    t = np.linspace(0, Ts, N, endpoint=False)
    
    f0 = -BW / 2
    k = BW / Ts
    upchirp = np.exp(1j * 2 * np.pi * (f0 * t + 0.5 * k * t**2))
    
    return upchirp, t, Ts

# Create figure
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Fixed time axis
t_fixed = np.linspace(0, fixed_time, int(fs * fixed_time), endpoint=False)

def animate(frame):
    SF = SF_values[frame % len(SF_values)]
    
    # Clear axes
    for ax in axes:
        ax.clear()
    
    # Generate chirp
    upchirp, t, Ts = generate_upchirp(SF, BW, fs)
    
    # Pad to fixed time window
    total_samples = int(fs * fixed_time)
    signal_padded = np.zeros(total_samples, dtype=complex)
    signal_padded[:len(upchirp)] = upchirp
    
    # Calculate metrics
    M = 2**SF
    data_rate = SF * BW / M
    
    # Plot 1: Time domain
    axes[0].plot(t_fixed * 1e3, np.real(signal_padded), 'b', linewidth=0.8)
    axes[0].axvline(x=Ts * 1e3, color='r', linestyle='--', linewidth=2, label=f'Symbol End: {Ts*1e3:.2f}ms')
    axes[0].fill_between(t_fixed * 1e3, -1.2, 1.2, where=(t_fixed <= Ts), alpha=0.2, color='blue')
    axes[0].set_xlim([0, fixed_time * 1e3])
    axes[0].set_ylim([-1.3, 1.3])
    axes[0].set_title(f'Time Domain | SF={SF} | Symbol Duration: {Ts*1e3:.2f}ms', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Amplitude')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Spectrogram with manual axis in ms
    nfft = 64
    if len(signal_padded) > nfft:
        Pxx, freqs, bins, im = axes[1].specgram(signal_padded, Fs=fs, NFFT=nfft, noverlap=60, cmap='viridis')
        
        # Convert bins from seconds to milliseconds
        bins_ms = bins * 1e3
        
        # Clear and replot with ms scale
        axes[1].clear()
        axes[1].pcolormesh(bins_ms, freqs / 1e3, 10 * np.log10(Pxx + 1e-10), cmap='viridis', shading='auto')
        axes[1].axvline(x=Ts * 1e3, color='r', linestyle='--', linewidth=2)
        axes[1].set_xlim([0, fixed_time * 1e3])
        axes[1].set_ylim([-BW/2/1e3, BW/2/1e3])
        axes[1].set_title('Spectrogram (Frequency Sweep)', fontsize=12)
        axes[1].set_xlabel('Time (ms)')
        axes[1].set_ylabel('Frequency (kHz)')
    
    # Plot 3: Bar chart comparison
    colors = ['green' if sf == SF else 'lightgray' for sf in SF_values]
    symbol_times = [(2**sf) / BW * 1e3 for sf in SF_values]
    bars = axes[2].bar([f'SF{sf}' for sf in SF_values], symbol_times, color=colors, edgecolor='black')
    axes[2].set_ylabel('Symbol Duration (ms)')
    axes[2].set_title(f'Time on Air Comparison | SF={SF}: {Ts*1e3:.2f}ms | Data Rate: {data_rate:.1f} bps', fontsize=12)
    axes[2].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, val in zip(bars, symbol_times):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                    f'{val:.2f}', ha='center', va='bottom', fontsize=10)
    
    fig.suptitle('LoRa Spreading Factor Trade-off: Higher SF = Longer Time on Air', 
                 fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    
    return axes

# Create animation
anim = FuncAnimation(fig, animate, frames=len(SF_values), interval=1000, repeat=True)

# Save as GIF
gif_path = 'lora_sf_tradeoff.gif'
writer = PillowWriter(fps=1)
anim.save(gif_path, writer=writer, dpi=100)
plt.close()

# Display the GIF
print(f"GIF saved as: {gif_path}")
print("\nSpreading Factor Trade-off:")
print("="*50)
for sf in SF_values:
    Ts = (2**sf) / BW
    rate = sf * BW / (2**sf)
    print(f"SF{sf}: Symbol = {Ts*1e3:6.2f}ms | Data Rate = {rate:6.1f} bps")
print("="*50)
print("\nHigher SF = More robust but SLOWER transmission!")

display(Image(filename=gif_path))

ModuleNotFoundError: No module named 'numpy'